In [ ]:
schema = ('acct_num', 'part_number', 'qty', 'amount', 'invoice_date', 'part_category', 'year')

In [ ]:
"""
Month over Month; even if a value is zero
"""

import pandas as pd


accts_tbl = pd.DataFrame(columns=['acct_num'], data=['STA810401'])
duckdb.sql("CREATE OR REPLACE TABLE accts_tbl AS SELECT * FROM accts_tbl")
duckdb.sql("INSERT INTO accts_tbl SELECT * FROM accts_tbl")

In [10]:
q = """
SELECT * FROM month_calendar
"""

df = conn.query(query=q).df()

df


,dates,month,year
0,2024-01-31,1,2024
1,2024-02-29,2,2024
2,2024-03-31,3,2024
3,2024-04-30,4,2024
4,2024-05-31,5,2024
...,...,...,...
163,2030-08-31,8,2030
164,2030-09-30,9,2030
165,2030-10-31,10,2030
166,2030-11-30,11,2030


In [ ]:
schema = ('acct_num', 'part_number', 'qty', 'amount', 'invoice_date', 'part_category', 'year')
 
q = """
WITH acct_base AS (
  SELECT 
    accts.acct_num,
    dates.month,
    dates.year  
  FROM accts_tbl accts
  CROSS JOIN dates_tbl AS dates
), 
grp_sales AS (
  SELECT
    s.acct_num,
    ROUND(SUM(s.amount)) AS month_net,
    EXTRACT (month FROM invoice_date) AS invoice_month, 
    EXTRACT (year FROM invoice_date) AS invoice_year
  FROM sales AS s
  GROUP BY 
    s.acct_num, 
    invoice_month, 
    invoice_year
),
month_totals AS (
SELECT 
  ab.acct_num,
  ab.month,
  ab.year,
  gs.month_net,
  gs.invoice_month,
  gs.invoice_year
FROM acct_base ab
LEFT JOIN grp_sales AS gs 
  ON  ab.acct_num = gs.acct_num
  AND ab.month    = gs.invoice_month
  AND ab.year     = gs.invoice_year 
)
PIVOT month_totals
ON year
USING sum(month_net)
"""

df = conn.query(query=q).df()

df


,acct_num,month,invoice_month,invoice_year,2024,2025
0,STA810401,8,8,2024,23066.0,NaN
1,STA810401,9,9,2025,NaN,16805.0
2,STA810401,10,10,2024,27952.0,NaN
3,STA810401,1,1,2025,NaN,23778.0
4,STA810401,7,7,2025,NaN,34490.0
5,STA810401,6,6,2024,18849.0,NaN
6,STA810401,4,4,2024,28317.0,NaN
7,STA810401,2,2,2024,33283.0,NaN
8,STA810401,3,3,2025,NaN,14284.0
9,STA810401,5,5,2024,8991.0,NaN


In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class SourceConfig:
    kind: str
    register_as: str
    path: str
    sheet: str
    header: int = 0
    
 
    def __post_init__(self):
        test_path = Path(self.path)

        assert test_path.exists() and test_path.is_file(), (
            f"src_path for {name} does not exist: got {self.path}"
        )


sources = {
    "sales_people": {
        "kind": "excel_sheet", 
        "path": r"H:\Sales Operations\dev_apps\sar_detail\2025_interface.xlsx",
        "sheet": "sales_people",
        "header": 0,
        "register_as": "raw_sales_people"
    },
}

for name, details  in sources.items():
    src = SourceConfig(**details)
    print(src.kind)
    print(src.path)
    print(src.sheet)
    print(src.header)
    print(src.register_as)

excel_sheet
H:\Sales Operations\dev_apps\sar_detail\2025_interface.xlsx
sales_people
0
raw_sales_people
